In [ ]:
import os
import numpy as np
from dask.distributed import Client
from dask import delayed
import dask.array as da
from skimage.transform import resize
from skimage.color import rgb2gray

from skimage.io import imread, imread_collection, imsave, ImageCollection

import warnings
warnings.filterwarnings('ignore') 

import matplotlib.pyplot as plt
%matplotlib inline

# Supervised Learning with Design Safe Data

Next, we try the same pipeline in the previous notebook, but with the DesignSafe Data. Let's see how our new models perform.

### Copy data for this notebook to your compute node

In [ ]:
! cp -r /work/00791/xwj/DMS/sc21-pdl/data/data.tar.gz /tmp/
! tar zxf /tmp/data.tar.gz -C /tmp

### Import data using EDA pipline from Day 1

In [ ]:
client = Client()

In [ ]:
from src.get_data_dask import get_all_data

classes = [0,2,4]
path_train = '/tmp/Dataset_2/Train/C{}/'
path_test = '/tmp/Dataset_2//Validation/C{}/'

path_list_train = [path_train.format(class_) for class_ in classes]
path_list_test = [path_test.format(class_) for class_ in classes]
        
train, test, y_train, y_test = get_all_data(path_list_train, path_list_test, classes, size=(112,112),gray=True)

In [ ]:
X_train = train.compute()

In [ ]:
X_test = test.compute()

In [ ]:
client.close()

In [ ]:
shape_train_all = X_train.shape
shape_test_all = X_test.shape 
shape_image = X_train[0].shape

In [ ]:
image_shape =  (112,112,3)

### Reshape data to classic feature and target

In [ ]:
X_train = X_train.reshape((len(X_train),-1))
X_test = X_test.reshape((len(X_test),-1))
X_train.shape,X_test.shape

### Functions for Model Evaluation 

In [ ]:
from sklearn.metrics import roc_curve, classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd

def plot_roc(model):
    """
    Plots multiclass (>2 labels) ROC curve
    """
    classes = ['low','medium','high']
    y_test_dummied = pd.get_dummies(y_test).values
    predicted_probs = model.predict_proba(X_test)

    fig,ax=plt.subplots()
    for i,class_ in enumerate(classes):
        fpr,tpr,thresholds = roc_curve(y_test_dummied[:,i],predicted_probs[:,i])
        ax.plot(fpr,tpr,label=class_)
        ax.plot([0,1],[0,1],color='k',ls='--')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
    ax.legend()

def evaluate(model, X_train, X_test, y_train, y_test, roc=True):
    """
    Computes and prints accuracy score on train and test data, and various metrics using testing data
    Plots the confusion matrix and ROC curve if requested
    """
    accuracy_test = accuracy_score(y_test, model.predict(X_test))
    print('Accuracy on the Test Data is {:.2f}'.format(accuracy_test))
    print('Accuracy on the Training Data is {:.2f}'.format(accuracy_score(y_train, model.predict(X_train))))
    if roc:
        plot_roc(model)
    cm_display = ConfusionMatrixDisplay.from_estimator(model, X_test, y_test,
                                                   cmap=plt.cm.Blues,normalize=None)
    return accuracy_test 

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score 

clf_dt = DecisionTreeClassifier()
clf_dt.fit(X_train, y_train)

In [ ]:
accuracy_dt = evaluate(clf_dt, X_train, X_test, y_train, y_test)

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
clf_lr = LogisticRegression(random_state=0).fit(X_train, y_train)

In [ ]:
accuracy_lr = evaluate(clf_lr, X_train, X_test, y_train, y_test)

### Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
y_pred = gnb.fit(X_train, y_train)

In [ ]:
accuracy_nb = evaluate(gnb, X_train, X_test, y_train, y_test, roc=False )

### Support Vector Machine Classifier

In [ ]:
#RBF Kernel normlize

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf_svm = SVC(C=10).fit(X_train_scaled, y_train)

In [ ]:
accuracy_svm = evaluate(clf_svm, X_train_scaled, X_test_scaled, y_train, y_test , roc=False)

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf_rf = RandomForestClassifier()
clf_rf.fit(X_train, y_train)

In [ ]:
accuracy_rf = evaluate(clf_rf, X_train, X_test, y_train, y_test )

In [ ]:
summary ='''
       Accuracy of all models: \n
       Decision Tree: {:.2f}\n
       Logistic Regression: {:.2f}\n
       Naive Bayes: {:.2f}\n
       Support Vector Machine: {:.2f} \n
       Random Forest {:.2f}
       '''.format(accuracy_dt, accuracy_lr, accuracy_nb, accuracy_svm, accuracy_rf)

print(summary)

Finally, lets take a look at what we got right and wrong in our highest performing model with a specific focus on the low versus high damage classes. 

In [ ]:
yhat = clf_rf.predict(X_test)
y_test = np.array(y_test)

misclassifications0 = X_test[ (y_test == 0) & (yhat == 4)]
misclassifications4 = X_test[ (y_test == 4) & (yhat == 0)]
misclassifications0.shape, misclassifications4.shape

In [ ]:
fig,ax = plt.subplots(2,2,figsize=(10,10))
for i,ax in enumerate(ax.flatten()):
    ax.imshow(misclassifications0[i].reshape(shape_image),cmap='gray')
fig.suptitle('Low Damage, But Predicted High');

In [ ]:
fig,ax = plt.subplots(2,2,figsize=(10,10))
for i,ax in enumerate(ax.flatten()):
    ax.imshow(misclassifications4[i].reshape(shape_image),cmap='gray')
fig.suptitle('High Damage, But Predicted Low');

In [ ]:
correct0 = X_test[ (y_test == 0) & (yhat == 0)]
correct4 = X_test[ (y_test == 4) & (yhat == 4)]
correct0.shape, correct4.shape

In [ ]:
fig,ax = plt.subplots(2,2,figsize=(10,10))
for i,ax in enumerate(ax.flatten()):
    ax.imshow(correct0[i].reshape(shape_image),cmap='gray')
fig.suptitle('Correctly Classified Low Damage');

In [ ]:
fig,ax = plt.subplots(2,2,figsize=(10,10))
for i,ax in enumerate(ax.flatten()):
    ax.imshow(correct4[i].reshape(shape_image),cmap='gray')
fig.suptitle('Correctly Classified High Damage');

In [ ]:
min_ = clf_rf.feature_importances_.min()
max_ = clf_rf.feature_importances_.max()

fig,ax = plt.subplots(figsize=(6,6))
cax = ax.imshow(clf_rf.feature_importances_.reshape((shape_image)),cmap='gray')
# Add colorbar, make sure to specify tick locations to match desired ticklabels
cbar = fig.colorbar(cax, ticks=[min_,max_])
cbar.ax.set_yticklabels(['low','high'])  # vertically oriented colorbar
cbar.ax.set_ylabel('Feature Importance')

**Questions** Is it obvious how our random forest model is making decisions here? 